In [1]:
from dotenv import load_dotenv
import os, json, re, math
from datetime import datetime, date

load_dotenv()

from openai import OpenAI
client = OpenAI()

print("Готово")

Готово


In [3]:
def tool_get_datetime() -> dict:
    """
    Возвращает текущую дату и время.
    Нет входных параметров.
    """
    now = datetime.now()
    days_map = {
        0: "понедельник", 1: "вторник", 2: "среда",
        3: "четверг",    4: "пятница", 5: "суббота", 6: "воскресенье"
    }
    months_map = {
        1: "января", 2: "февраля", 3: "марта",    4: "апреля",
        5: "мая",    6: "июня",    7: "июля",      8: "августа",
        9: "сентября", 10: "октября", 11: "ноября", 12: "декабря"
    }
    return {
        "iso": now.strftime("%Y-%m-%d"),
        "day_of_week": days_map[now.weekday()],
        "day": now.day,
        "month": now.month,
        "month_name": months_map[now.month],
        "year": now.year,
        "time": now.strftime("%H:%M:%S"),
        "readable": f"{now.day} {months_map[now.month]} {now.year} года, {days_map[now.weekday()]}"
    }


def tool_calculator(expression: str) -> dict:
    """
    Вычисляет математическое выражение.
    Поддерживает: +, -, *, /, **, sqrt(), abs(), round(), ceil(), floor(),
    log(), log10(), sin(), cos(), tan(), pi, e, days_until(YYYY-MM-DD).
    Пример: 'sqrt(341)', '2 ** 10', 'days_until(2026-01-01)'
    """
    try:
        days_match = re.search(r'days_until\((\d{4}-\d{2}-\d{2})\)', expression)
        if days_match:
            target = date.fromisoformat(days_match.group(1))
            delta = (target - date.today()).days
            return {"expression": expression, "result": delta, "note": f"дней до {target}"}

        safe_env = {
            "__builtins__": {},
            "sqrt": math.sqrt, "abs": abs, "round": round,
            "ceil": math.ceil, "floor": math.floor,
            "log": math.log, "log10": math.log10,
            "sin": math.sin, "cos": math.cos, "tan": math.tan,
            "pi": math.pi, "e": math.e, "pow": pow,
        }
        result = eval(expression, safe_env)  
        return {"expression": expression, "result": round(result, 6)}

    except Exception as ex:
        return {"expression": expression, "error": str(ex)}


TOOLS = {
    "get_datetime": tool_get_datetime,
    "calculator":   tool_calculator,
}

print("Дата:", tool_get_datetime())
print("sqrt(144):", tool_calculator("sqrt(144)"))
print("days_until:", tool_calculator("days_until(2027-01-01)"))

Дата: {'iso': '2026-05-09', 'day_of_week': 'суббота', 'day': 9, 'month': 5, 'month_name': 'мая', 'year': 2026, 'time': '03:43:41', 'readable': '9 мая 2026 года, суббота'}
sqrt(144): {'expression': 'sqrt(144)', 'result': 12.0}
days_until: {'expression': 'days_until(2027-01-01)', 'result': 237, 'note': 'дней до 2027-01-01'}


In [4]:
REACT_SYSTEM_PROMPT = """
Ты — интеллектуальный помощник. Для ответа на вопрос ты можешь рассуждать
и вызывать инструменты, следуя строгому формату ReAct.

== ДОСТУПНЫЕ ИНСТРУМЕНТЫ ==

1. get_datetime
   Описание: возвращает текущую дату, время и день недели.
   Вызов: нет параметров.

2. calculator
   Описание: вычисляет математическое выражение.
   Параметр expression — строка с формулой.
   Примеры выражений:
     - "sqrt(341)"              → корень из 341
     - "2 ** 10"                → степень
     - "days_until(2026-01-01)" → дней до указанной даты
     - "(100 - 32) / 1.8"      → арифметика
     - "log(100, 10)"           → логарифм

== ФОРМАТ ОТВЕТА ==

На каждом шаге выводи РОВНО один блок из двух вариантов:

Вариант A — если нужен инструмент:
```
Мысль: <рассуждение, что нужно сделать>
Действие: <имя_инструмента>
Параметры: <JSON-объект с параметрами, или {} если параметров нет>
```

Вариант B — если уже есть финальный ответ:
```
Мысль: <итоговое рассуждение>
Финальный ответ: <ответ пользователю>
```

== ПРАВИЛА ==
- НИКОГДА не вызывай инструмент и не давай финальный ответ в одном блоке.
- НИКОГДА не придумывай результаты инструментов — жди Наблюдение.
- Используй инструменты столько раз, сколько нужно.
- Если инструмент вернул ошибку — попробуй исправить параметры и повтори.
""".strip()

In [5]:
def parse_agent_response(text: str) -> dict:
    """
    Разбирает текст ответа LLM.
    Возвращает dict с ключами:
      type: 'action' | 'final'
      thought: str
      tool / params  (для action)
      answer         (для final)
    """
    thought_match = re.search(r'Мысль:\s*(.+?)(?=\n(?:Действие|Финальный))', text, re.DOTALL)
    thought = thought_match.group(1).strip() if thought_match else ""

    final_match = re.search(r'Финальный ответ:\s*(.+)', text, re.DOTALL)
    if final_match:
        return {"type": "final", "thought": thought, "answer": final_match.group(1).strip()}

    action_match = re.search(r'Действие:\s*(\w+)', text)
    params_match = re.search(r'Параметры:\s*(\{.*?\})', text, re.DOTALL)

    if action_match:
        tool_name = action_match.group(1).strip()
        params = {}
        if params_match:
            try:
                params = json.loads(params_match.group(1))
            except json.JSONDecodeError:
                params = {}
        return {"type": "action", "thought": thought, "tool": tool_name, "params": params}

    return {"type": "final", "thought": thought, "answer": text.strip()}

In [6]:
def run_react_agent(user_query: str, model: str = "gpt-4o-mini", max_steps: int = 10) -> str:
    """
    Запускает агент ReAct для заданного вопроса.
    Возвращает финальный ответ.
    """
    print("=" * 60)
    print(f"ВОПРОС: {user_query}")
    print("=" * 60)

    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT},
        {"role": "user",   "content": user_query},
    ]

    total_tokens = 0

    for step in range(1, max_steps + 1):
        print(f"\n--- Шаг #{step} ---")

        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.0,
            max_tokens=512,
        )
        total_tokens += response.usage.total_tokens
        raw = response.choices[0].message.content

        messages.append({"role": "assistant", "content": raw})

        parsed = parse_agent_response(raw)

        print(f"💭 Мысль: {parsed['thought']}")

        if parsed["type"] == "final":
            print(f"\n✅ Финальный ответ: {parsed['answer']}")
            print(f"\n📊 Всего токенов использовано: {total_tokens}")
            print("=" * 60)
            return parsed["answer"]

        tool_name = parsed["tool"]
        params    = parsed["params"]

        print(f"🔧 Действие: {tool_name}({params})")

        if tool_name not in TOOLS:
            observation = json.dumps({"error": f"Инструмент '{tool_name}' не найден. Доступны: {list(TOOLS.keys())}"}, ensure_ascii=False)
        else:
            fn = TOOLS[tool_name]
            try:
                result = fn(**params) if params else fn()
                observation = json.dumps(result, ensure_ascii=False)
            except Exception as e:
                observation = json.dumps({"error": str(e)}, ensure_ascii=False)

        print(f"👁  Наблюдение: {observation}")

        messages.append({"role": "user", "content": f"Наблюдение: {observation}"})

    print("⚠️  Достигнут лимит шагов")
    return "Не удалось получить ответ в рамках лимита шагов."

## Тест 1 — Сценарий из задания
**«Сколько дней осталось до Нового года и каков корень из этого числа?»**

In [7]:
answer1 = run_react_agent(
    "Сколько дней осталось до Нового года (1 января следующего года) "
    "и каков квадратный корень из этого числа? Округли до 2 знаков."
)

ВОПРОС: Сколько дней осталось до Нового года (1 января следующего года) и каков квадратный корень из этого числа? Округли до 2 знаков.

--- Шаг #1 ---
💭 Мысль: Сначала нужно узнать, сколько дней осталось до Нового года (1 января следующего года). Для этого я вызову инструмент, который вычисляет количество дней до указанной даты.
🔧 Действие: calculator({'expression': 'days_until(2024-01-01)'})
👁  Наблюдение: {"expression": "days_until(2024-01-01)", "result": -859, "note": "дней до 2024-01-01"}

--- Шаг #2 ---
💭 Мысль: Похоже, что я ошибся с датой, так как результат отрицательный. Нужно проверить текущую дату и пересчитать количество дней до Нового года. Сначала я вызову инструмент, чтобы получить текущую дату.
🔧 Действие: get_datetime({})
👁  Наблюдение: {"iso": "2026-05-09", "day_of_week": "суббота", "day": 9, "month": 5, "month_name": "мая", "year": 2026, "time": "03:46:35", "readable": "9 мая 2026 года, суббота"}

--- Шаг #3 ---
💭 Мысль: Текущая дата — 9 мая 2026 года. Это значит, что

## Тест 2 — Комбинированный запрос
**«Какой сегодня день недели? Если это будний день, возведи номер дня в степень 3.»**

In [8]:
answer2 = run_react_agent(
    "Какой сегодня день недели и какой это день по счёту в году? "
    "Возведи номер дня в квадрат и найди его квадратный корень."
)

ВОПРОС: Какой сегодня день недели и какой это день по счёту в году? Возведи номер дня в квадрат и найди его квадратный корень.

--- Шаг #1 ---
💭 Мысль: Сначала мне нужно узнать текущую дату, чтобы определить день недели и номер дня в году.
🔧 Действие: get_datetime({})
👁  Наблюдение: {"iso": "2026-05-09", "day_of_week": "суббота", "day": 9, "month": 5, "month_name": "мая", "year": 2026, "time": "03:47:58", "readable": "9 мая 2026 года, суббота"}

--- Шаг #2 ---
💭 Мысль: Сегодня 9 мая 2026 года, это суббота. Теперь мне нужно возвести номер дня (9) в квадрат и найти его квадратный корень.
🔧 Действие: calculator({'expression': 'sqrt(9 ** 2)'})
👁  Наблюдение: {"expression": "sqrt(9 ** 2)", "result": 9.0}

--- Шаг #3 ---
💭 Мысль: Я возвел номер дня (9) в квадрат и нашел его квадратный корень, который равен 9.

✅ Финальный ответ: Сегодня суббота, 9 мая 2026 года, номер дня в году — 9, его квадрат — 81, а квадратный корень из него — 9.
```

📊 Всего токенов использовано: 1864


## Тест 3 — Только калькулятор
**«Чему равно (2^32) / 1024 / 1024 и как называется этот объем памяти?»**

In [9]:
answer3 = run_react_agent(
    "Вычисли: сколько мегабайт в 2^32 байтах? Используй калькулятор."
)

ВОПРОС: Вычисли: сколько мегабайт в 2^32 байтах? Используй калькулятор.

--- Шаг #1 ---
💭 Мысль: Чтобы узнать, сколько мегабайт в 2^32 байтах, нужно разделить 2^32 на 1024^2, так как 1 мегабайт равен 1024^2 байтам.
🔧 Действие: calculator({'expression': '2 ** 32 / (1024 ** 2)'})
👁  Наблюдение: {"expression": "2 ** 32 / (1024 ** 2)", "result": 4096.0}

--- Шаг #2 ---
💭 Мысль: Я получил результат вычисления, который показывает, что в 2^32 байтах содержится 4096 мегабайт.

✅ Финальный ответ: В 2^32 байтах содержится 4096 мегабайт.

📊 Всего токенов использовано: 1117


---
## Интерактивный режим

Cвой вопрос:

In [10]:
my_question = "Сколько минут прошло с начала текущего года до сегодняшнего дня?"

answer = run_react_agent(my_question)

ВОПРОС: Сколько минут прошло с начала текущего года до сегодняшнего дня?

--- Шаг #1 ---
💭 Мысль: Сначала нужно узнать текущую дату и время, чтобы определить, сколько минут прошло с начала текущего года.
🔧 Действие: get_datetime({})
👁  Наблюдение: {"iso": "2026-05-09", "day_of_week": "суббота", "day": 9, "month": 5, "month_name": "мая", "year": 2026, "time": "03:49:23", "readable": "9 мая 2026 года, суббота"}

--- Шаг #2 ---
💭 Мысль: Теперь, когда у нас есть текущая дата (9 мая 2026 года), нужно вычислить, сколько минут прошло с начала года (1 января 2026 года) до этой даты. Для этого сначала найдем количество дней между этими датами, а затем переведем это количество в минуты.
🔧 Действие: calculator({'expression': '(9 - 1) + 30 + 31 + 28 + 31 + 30 + 31 + 31 + 30 + 31 + 30 + 31'})
👁  Наблюдение: {"expression": "(9 - 1) + 30 + 31 + 28 + 31 + 30 + 31 + 31 + 30 + 31 + 30 + 31", "result": 342}

--- Шаг #3 ---
💭 Мысль: Теперь, когда мы знаем, что прошло 342 дня с начала года, нужно перевести